## MMR (Maximal Marginal Relevance)

It is a powerful diversity-aware retrieval techinique used in information retrievel and RAG pipeline to balance relevance and novelty when selecting documents.


MMR selects documents that are both : 
1. Relevant to the query.
2. Diverse from each other.

It prevents the retriever form returning very similar documents that repeat the same content.

MMR=lamda x similarity( doc ,Q ) - ( 1-lambda ) x Max(s) x similarty(doc,s)
lambda is tunable which signfies on left side how much weightage to be given to left side i.e. lambda x something which tell weightage to relevance.

while right side (1-lambda) x something signifies how diverse the output should nbe

In [4]:
with open("mmr.txt","r")as f:
    content=f.read()
    
chunks = [c.strip() for c in content.split("===") if c.strip() and not c.strip().startswith("CHUNK")]


'# LangChain Knowledge Base - Sample Document for MMR Testing\n# This file contains intentionally repetitive content to demonstrate MMR effectiveness'

In [11]:
from langchain_openai import OpenAIEmbeddings,ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain

In [19]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

In [8]:
embeddings=OpenAIEmbeddings(model="text-embedding-3-small")


vectorstore=FAISS.from_texts(chunks,embeddings)

In [14]:
retriever=vectorstore.as_retriever(search_kwargs={"k": 5})

retrieved_docs=retriever.invoke("What is LangChain and how does RAG work?")

for i, doc in enumerate(retrieved_docs):
    print(f"--- Doc {i+1} ---")
    print(doc.page_content)
    print()

--- Doc 1 ---
In LangChain, RAG pipelines retrieve external documents from a vector store and provide them
as context to the language model. This helps the LLM answer questions using knowledge that
wasn't part of its original training data.

--- Doc 2 ---
RAG (Retrieval-Augmented Generation) is a technique in LangChain where external knowledge
is retrieved from a vector store and passed to the LLM as context. This allows the model
to answer questions beyond its training data by fetching relevant documents at runtime.

--- Doc 3 ---
Retrieval-Augmented Generation, or RAG, is a core LangChain pattern. It works by retrieving
relevant documents from a vector database and injecting them into the LLM prompt as context,
enabling the model to use up-to-date or domain-specific knowledge.

--- Doc 4 ---
LangChain is a popular open-source framework that helps developers build applications
powered by large language models. It offers a unified interface for working with chains,
agents, and memory c

In [16]:
mmr_retriever=vectorstore.as_retriever(search_kwargs={"k": 5,
        "fetch_k": 15,
        "lambda_mult": 0.7 },search_type="mmr")
mmr_retrieved_docs=mmr_retriever.invoke("What is LangChain and how does RAG work?")

for i, doc in enumerate(mmr_retrieved_docs):
    print(f"--- MMR Doc {i+1} ---")
    print(doc.page_content)
    print()

--- MMR Doc 1 ---
In LangChain, RAG pipelines retrieve external documents from a vector store and provide them
as context to the language model. This helps the LLM answer questions using knowledge that
wasn't part of its original training data.

--- MMR Doc 2 ---
LangChain is a popular open-source framework that helps developers build applications
powered by large language models. It offers a unified interface for working with chains,
agents, and memory components in LLM applications.

--- MMR Doc 3 ---
Retrieval-Augmented Generation, or RAG, is a core LangChain pattern. It works by retrieving
relevant documents from a vector database and injecting them into the LLM prompt as context,
enabling the model to use up-to-date or domain-specific knowledge.

--- MMR Doc 4 ---
The LangChain Expression Language lets developers compose modular pipelines using a pipe
syntax. Components such as retrievers, prompt templates, LLMs, and parsers can be chained
together cleanly and support parallel exe

In [24]:
dense_vector_store = FAISS.from_texts(chunks, embeddings)
dense_retriever = dense_vector_store.as_retriever(search_kwargs={"k": 3})

sparse_retriever = BM25Retriever.from_texts(chunks)
sparse_retriever.k = 3  # top k documents to retrieve

hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.7, 0.3],
)

In [26]:
hybrid_docs = hybrid_retriever.invoke("What is LangChain and how does RAG work?")

for i, doc in enumerate(hybrid_docs):
    print(f"--- Hybrid Doc {i+1}")
    print(doc.page_content)
    print()

--- Hybrid Doc 1
RAG (Retrieval-Augmented Generation) is a technique in LangChain where external knowledge
is retrieved from a vector store and passed to the LLM as context. This allows the model
to answer questions beyond its training data by fetching relevant documents at runtime.

--- Hybrid Doc 2
In LangChain, RAG pipelines retrieve external documents from a vector store and provide them
as context to the language model. This helps the LLM answer questions using knowledge that
wasn't part of its original training data.

--- Hybrid Doc 3
Retrieval-Augmented Generation, or RAG, is a core LangChain pattern. It works by retrieving
relevant documents from a vector database and injecting them into the LLM prompt as context,
enabling the model to use up-to-date or domain-specific knowledge.

--- Hybrid Doc 4
LangChain was created to make it easier to develop LLM-based applications. The framework
standardizes how developers interact with language models, providing tools for chaining,
agent

In [30]:
# increase candidate pool
dense_retriever = dense_vector_store.as_retriever(search_kwargs={"k": 10})

sparse_retriever = BM25Retriever.from_texts(chunks)
sparse_retriever.k = 10  # increase this too

hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.7, 0.3],
)

# now run the combined pipeline
final_docs = hybrid_mmr_retriever(
    "What is LangChain and how does RAG work?",
    fetch_k=15,
    final_k=5,
    lambda_mult=0.7
)

for i, doc in enumerate(final_docs):
    print(f"--- Final Doc {i+1}  ---")
    print(doc.page_content)
    print()

--- Final Doc 1  ---
In LangChain, RAG pipelines retrieve external documents from a vector store and provide them
as context to the language model. This helps the LLM answer questions using knowledge that
wasn't part of its original training data.

--- Final Doc 2  ---
LangChain is a popular open-source framework that helps developers build applications
powered by large language models. It offers a unified interface for working with chains,
agents, and memory components in LLM applications.

--- Final Doc 3  ---
Retrieval-Augmented Generation, or RAG, is a core LangChain pattern. It works by retrieving
relevant documents from a vector database and injecting them into the LLM prompt as context,
enabling the model to use up-to-date or domain-specific knowledge.

--- Final Doc 4  ---
The LangChain Expression Language lets developers compose modular pipelines using a pipe
syntax. Components such as retrievers, prompt templates, LLMs, and parsers can be chained
together cleanly and support 